# DiodeMCP Track A — `03_export_and_bench.ipynb`
**Eager ONNX `opset=17` Export, `MatMul`/`Gemm`-Only INT8 Quantization, Post-INT8 Temperature Scaling, `DiodeModelBundle v1` Packaging & Benchmark Suite**

Packages and benchmarks all 3 trained `token_roles`-isolated Jev cross-encoders into `/content/drive/MyDrive/diodemcp/bundles/`:
- **`DiodeMCP-Jev-Base-149M`** (`answerdotai/ModernBERT-base`)
- **`DiodeMCP-Jev-Large-395M`** (`answerdotai/ModernBERT-large`)
- **`DiodeMCP-NanoJev-0.6B`** (`Qwen/Qwen3-0.6B-Base`)

**Pipeline (`RA3`, `R5`, `R6`, `I7`, `I8`, `I11`):**
1. Exports eager `opset=17` ONNX with internal 4D `token_roles` mask and static output widths pinned (`pin_onnx_output_shapes`).
2. Quantizes ONLY backbone `MatMul`/`Gemm` weight projections to INT8 (`model_int8.onnx`), keeping `LayerNorm`, `Softmax`, 4D mask ops, and `HEAD_SCOPE_NAMES` (`/noul_head`, `/src_pointer`, `/arg_pointer`, `/logit_temperatures`) in pure FP32.
3. Runs `val_calib` through the **quantized INT8 ORT session**, fits `(T_enc, T_sem, T_hijack)` via `fit_temperature_scaling`, bakes the FP32 temperatures into `model_int8.onnx`, writes `calibration.json` + `tokenizer.json` + `manifest.json` (`sha256_model_onnx`, `sha256_tokenizer_json`, `sha256_calibration_json`), and validates each bundle via `validate_bundle_dir`.
4. Runs `run_bundle_benchmark` and `format_benchmark_comparison_table` across `val_calib`, `trifecta_test`, and CPU single-request latency (`512` & `2048` buckets).

In [ ]:
# 1. Mount Google Drive, extract model_lab_src.tar.gz if needed (zero GitHub PAT), & install model-lab
import json
import os
import subprocess
import tarfile
from pathlib import Path

try:
    from google.colab import drive  # type: ignore[import-not-found]
    drive.mount("/content/drive", force_remount=False)
except ImportError:
    print("Running outside Google Colab runtime; skipping drive.mount().")

!pip install -q uv

DRIVE_ROOT = Path(os.environ.get("DIODEMCP_DRIVE_ROOT", "/content/drive/MyDrive/diodemcp"))
SRC_TARBALL = DRIVE_ROOT / "model_lab_src.tar.gz"
EXTRACT_TARGET = Path("/content/diodemcp")

CANDIDATE_MODEL_LAB_DIRS = (
    Path("/content/diodemcp/model-lab"),
    Path("/content/model-lab"),
    DRIVE_ROOT / "model-lab",
    Path("./model-lab"),
    Path("../model-lab"),
    Path("."),
)

def _find_model_lab_dir() -> Path | None:
    return next(
        (
            cand.resolve()
            for cand in CANDIDATE_MODEL_LAB_DIRS
            if (cand / "pyproject.toml").is_file() and (cand / "src" / "diodemodel").is_dir()
        ),
        None,
    )

MODEL_LAB_DIR = _find_model_lab_dir()
if MODEL_LAB_DIR is None and SRC_TARBALL.is_file():
    print(f"Extracting {SRC_TARBALL} -> {EXTRACT_TARGET} ...")
    EXTRACT_TARGET.mkdir(parents=True, exist_ok=True)
    with tarfile.open(SRC_TARBALL, "r:gz") as tar:
        tar.extractall(path=EXTRACT_TARGET)
    MODEL_LAB_DIR = _find_model_lab_dir()

if MODEL_LAB_DIR is None:
    raise FileNotFoundError(
        f"Could not locate model-lab/pyproject.toml in {[str(c) for c in CANDIDATE_MODEL_LAB_DIRS]} "
        f"and archive {SRC_TARBALL} was not found."
    )

print(f"Installing model-lab from: {MODEL_LAB_DIR}")
subprocess.run(["uv", "pip", "install", "--system", "-e", str(MODEL_LAB_DIR)], check=True)

In [ ]:
# 2. Configure Drive layout (/content/drive/MyDrive/diodemcp/{datasets,checkpoints,bundles}) & Model Matrix
from tokenizers import Tokenizer

from diodemodel.benchmark_suite import (
    DEFAULT_LATENCY_BUCKETS,
    format_benchmark_comparison_table,
    run_bundle_benchmark,
)
from diodemodel.contract import REQUIRED_SPECIAL_TOKENS, validate_bundle_dir
from diodemodel.export_bundle import export_diode_bundle
from diodemodel.modeling_jev import ModernBERTJevModel, NanoJevCausalModel, _JevModelBase
from diodemodel.tokenizer_builder import build_jev_tokenizer
from diodemodel.train import (
    CURRICULUM_BUCKETS,
    load_checkpoint,
    log_train_event,
    make_synthetic_curriculum_dataset,
)

DATASETS_DIR = DRIVE_ROOT / "datasets"
TOKENIZERS_DIR = DATASETS_DIR / "tokenizers"
CHECKPOINTS_DIR = Path(os.environ.get("DIODEMCP_CHECKPOINTS_DIR", "/content/drive/MyDrive/diodemcp/checkpoints"))
BUNDLES_DIR = Path(os.environ.get("DIODEMCP_BUNDLES_DIR", "/content/drive/MyDrive/diodemcp/bundles"))
GLOBAL_STATUS_LOG = CHECKPOINTS_DIR / "live_train_status.jsonl"

for directory in (DATASETS_DIR, TOKENIZERS_DIR, CHECKPOINTS_DIR, BUNDLES_DIR):
    directory.mkdir(parents=True, exist_ok=True)

SMOKE_MODE = os.environ.get("DIODEMCP_SMOKE_MODE", "0") == "1"
INTRA_OP_THREADS = int(os.environ.get("DIODEMCP_ORT_THREADS", "4"))

MODEL_MATRIX = (
    {
        "model_id": "DiodeMCP-Jev-Base-149M",
        "profile_key": "modernbert_base",
        "hf_backbone": "answerdotai/ModernBERT-base",
        "arch": "modernbert",
    },
    {
        "model_id": "DiodeMCP-Jev-Large-395M",
        "profile_key": "modernbert_large",
        "hf_backbone": "answerdotai/ModernBERT-large",
        "arch": "modernbert",
    },
    {
        "model_id": "DiodeMCP-NanoJev-0.6B",
        "profile_key": "qwen3_0_6b",
        "hf_backbone": "Qwen/Qwen3-0.6B-Base",
        "arch": "nanojev",
    },
)

log_train_event(
    GLOBAL_STATUS_LOG,
    "notebook_03_initialized",
    bundles_dir=str(BUNDLES_DIR),
    smoke_mode=SMOKE_MODE,
    intra_op_threads=INTRA_OP_THREADS,
)

In [ ]:
# 3. Load existing Jev tokenizers from Drive (reusing 01_dataset_factory / 02_train_jev_models tokenizers)
loaded_tokenizers: dict[str, tuple[Path, Tokenizer, dict[str, int]]] = {}
for spec in MODEL_MATRIX:
    profile_key = str(spec["profile_key"])
    hf_backbone = str(spec["hf_backbone"])
    tok_dir = TOKENIZERS_DIR / profile_key
    tok_json_path = tok_dir / "tokenizer.json"
    if tok_json_path.is_file():
        rust_tok = Tokenizer.from_file(str(tok_json_path))
        special_ids = {
            tok_str: int(rust_tok.token_to_id(tok_str))
            for tok_str in REQUIRED_SPECIAL_TOKENS
        }
    else:
        special_ids = build_jev_tokenizer(hf_backbone, tok_dir)
        rust_tok = Tokenizer.from_file(str(tok_json_path))
    loaded_tokenizers[profile_key] = (tok_json_path, rust_tok, special_ids)
    print(f"[{profile_key}] tokenizer={tok_json_path} vocab_size={rust_tok.get_vocab_size()}")

In [ ]:
# 4. Export opset-17 ONNX, MatMul/Gemm-only INT8 quantize, post-INT8 temperature calibrate & validate bundle
def export_and_benchmark_model_spec(spec: dict[str, object]) -> dict[str, object]:
    model_id = str(spec["model_id"])
    profile_key = str(spec["profile_key"])
    hf_backbone = str(spec["hf_backbone"])
    arch = str(spec["arch"])

    tok_json_path, rust_tok, special_ids = loaded_tokenizers[profile_key]
    vocab_size = int(rust_tok.get_vocab_size())
    pad_id = int(special_ids["[PAD]"])

    model_ckpt_dir = CHECKPOINTS_DIR / model_id
    best_ckpt = model_ckpt_dir / "checkpoint_best.pt"
    latest_ckpt = model_ckpt_dir / "checkpoint_latest.pt"
    ckpt_path = best_ckpt if best_ckpt.is_file() else latest_ckpt

    model_cls = NanoJevCausalModel if arch == "nanojev" else ModernBERTJevModel
    model: _JevModelBase = model_cls.from_pretrained(hf_backbone, vocab_size=vocab_size)
    if ckpt_path.is_file():
        load_checkpoint(ckpt_path, model=model, restore_rng=False, map_location="cpu")
        print(f"[{model_id}] Loaded trained weights from {ckpt_path}")
    elif not SMOKE_MODE:
        raise FileNotFoundError(f"Trained checkpoint not found for {model_id} in {model_ckpt_dir}")

    profile_data_dir = DATASETS_DIR / profile_key
    val_calib_pq = profile_data_dir / "diodebench_mcp_250k_val_calib.parquet"
    trifecta_test_pq = profile_data_dir / "trifecta_dev_bench_5k_eval.parquet"

    if SMOKE_MODE or not val_calib_pq.is_file():
        val_calib_source: object = make_synthetic_curriculum_dataset(
            16, vocab_size=min(vocab_size, 256), seed=101, pad_token_id=pad_id
        )
        trifecta_test_source: object = make_synthetic_curriculum_dataset(
            16, vocab_size=min(vocab_size, 256), seed=202, pad_token_id=pad_id
        )
    else:
        val_calib_source = val_calib_pq
        trifecta_test_source = trifecta_test_pq if trifecta_test_pq.is_file() else val_calib_pq

    bundle_dir = BUNDLES_DIR / model_id
    export_res = export_diode_bundle(
        model,
        bundle_dir,
        tokenizer_source=tok_json_path,
        model_id=model_id,
        val_calib_rows=val_calib_source,  # type: ignore[arg-type]
        special_token_ids=special_ids,
        calib_buckets=CURRICULUM_BUCKETS,
        intra_op_threads=INTRA_OP_THREADS,
    )
    validated_manifest = validate_bundle_dir(bundle_dir, verify_sha256=True)
    print(
        f"[{model_id}] Validated DiodeModelBundle v1 at {bundle_dir} "
        f"(sha256_model_onnx={validated_manifest.sha256_model_onnx[:12]}... "
        f"post-INT8 ece_val={validated_manifest.ece_val:.4f})"
    )

    bench_report = run_bundle_benchmark(
        bundle_dir,
        val_rows=val_calib_source,  # type: ignore[arg-type]
        trifecta_rows=trifecta_test_source,  # type: ignore[arg-type]
        latency_buckets=DEFAULT_LATENCY_BUCKETS,
        num_latency_runs=3 if SMOKE_MODE else 10,
        warmup_runs=1 if SMOKE_MODE else 2,
        intra_op_threads=INTRA_OP_THREADS,
        output_json_path=bundle_dir / "benchmark_report.json",
    )
    log_train_event(
        GLOBAL_STATUS_LOG,
        "model_bundle_exported_and_benchmarked",
        model_id=model_id,
        bundle_dir=str(bundle_dir),
        sha256_model_onnx=validated_manifest.sha256_model_onnx,
        temperatures=export_res["calibration"]["temperatures"],
        ece_val=bench_report["ece_val"],
        attack_recall=bench_report["attack_recall"],
        flipjev_attack_recall=bench_report["flipjev_attack_recall"],
        flipjev_benign_dos_fpr=bench_report["flipjev_benign_dos_fpr"],
        trifecta_safe_fpr=bench_report["trifecta_safe_fpr"],
        latency_512_p50_ms=bench_report["latency_512_p50_ms"],
        latency_2048_p50_ms=bench_report["latency_2048_p50_ms"],
    )
    return bench_report

In [ ]:
# 5. Export, INT8-Quantize, Post-INT8 Calibrate & Benchmark All 3 Jev Models
benchmark_reports: list[dict[str, object]] = []
for model_spec in MODEL_MATRIX:
    rep = export_and_benchmark_model_spec(model_spec)
    benchmark_reports.append(rep)

In [ ]:
# 6. Verify All 3 Bundle Directories & Write Comparison Table (Markdown + JSON)
for model_spec in MODEL_MATRIX:
    m_id = str(model_spec["model_id"])
    validate_bundle_dir(BUNDLES_DIR / m_id, verify_sha256=True)

comparison_md = format_benchmark_comparison_table(benchmark_reports)  # type: ignore[arg-type]
print("\n=== DiodeMCP v1 INT8 Bundle Benchmark Comparison ===\n")
print(comparison_md)

(BUNDLES_DIR / "benchmark_comparison_table.md").write_text(comparison_md, encoding="utf-8")
(BUNDLES_DIR / "benchmark_comparison_report.json").write_text(
    json.dumps(
        {
            "reports": benchmark_reports,
            "comparison_table_markdown": comparison_md,
        },
        indent=2,
        sort_keys=True,
    ),
    encoding="utf-8",
)
print(f"Saved bundle comparison report to {BUNDLES_DIR / 'benchmark_comparison_report.json'}")